In [0]:
import os
#restart python
dbutils.library.restartPython()

In [0]:
import sys
sys.path.append("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice")
from utils.config_reader import ConfigReader
from utils.rename_column_names import ColumnMapping
from utils.datatype_update import DatatypeUpdate
from utils.audit import AuditManager

from datetime import datetime

In [0]:
#read config file
config=ConfigReader.read_config("/Workspace/Users/nalluriravi3@gmail.com/New_Medalion_Practice/config/bronze_config.json")
customer_config=config["customer"]



In [0]:
#create source and target data frame
source_df=spark.read.format(customer_config["source_format"])\
               .option("delimeter",customer_config["delimiter"])\
               .option("header",customer_config["header"])\
               .load(customer_config["source_path"])

target_df=spark.table(f"{customer_config['target_catalog']}."
                     f"{customer_config['target_schema']}."
                     f"{customer_config['target_table']}"
                     )
           

## Handle schema column name & data type changes

In [0]:
# verify target columns and source columns and modify if there is a change
#mapping=customer_config.get("rename_mapping",{})

source_df=ColumnMapping.col_rename(source_df,customer_config.get("rename_mapping",{}))

source_df=DatatypeUpdate.update_datatype(source_df,target_df)

source_df.display()



## Load data into bronze_customer table

In [0]:
from datetime import datetime
from pyspark.sql.functions import current_timestamp

start_time = datetime.now()
status = ""
error_message = ""
records_rejected = 0
records_written = 0
records_read = source_df.count()

source_df = source_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
)
source_df.display()
try:
    
    source_df.write.format("delta")\
        .mode("append")\
        .option("mergeSchema", customer_config["schema_evolution"])\
        .saveAsTable(f"{customer_config['target_catalog']}."
                     f"{customer_config['target_schema']}."
                     f"{customer_config['target_table']}")
    status = "SUCCESS"
    records_rejected = 0
    records_written=records_read
    
except Exception as e:
    status = "FAIL"
    records_written = 0
    error_message = str(e)
    raise e

finally:
    AuditManager.log_audit(
        spark=spark,
        pipeline_name="customer_pipeline",
        layer_name="bronze",
        table_name=customer_config["target_table"],
        start_time=start_time,
        end_time=datetime.now(),
        records_read=records_read,
        records_written=records_written,
        records_rejected=records_rejected,
        status=status,
        error_message=error_message
    )

if error_message:
    dbutils.notebook.exit(
        f"FAILED : {error_message}"
    )
else:
    dbutils.notebook.exit("SUCCESS")
        